# HDMI 自环测试：板子 OUT → 板子 IN

**这一步不需要笔记本，也不需要采集卡。**

## 接线（重要）

1. **拔掉**接在板子 HDMI IN 上的笔记本线
2. **拔掉**接在板子 HDMI OUT 上的采集卡线
3. 用**同一根线**，把板子的 **HDMI OUT** 直接插到 **HDMI IN**

也就是：板子自己输出、自己接收，形成闭环。

## 然后 Run All

看最后一行输出：

- `LOCKED 1920x1080@60` → **板子的 HDMI 输入硬件是好的**，问题在笔记本送出的信号
- `still None` → **板子这侧的输入通道有问题**（接头/PHY），跟笔记本无关


In [ ]:
import time
from pynq import Overlay
from pynq.lib.video import VideoMode

STOCK = "/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/overlays/base/base.bit"
ol = Overlay(STOCK)
v = ol.video
print("stock overlay loaded, IPs:", len(ol.ip_dict))

# The stock dvi2rgb is configured for TMDS >= 120 MHz, so drive the output at
# 1080p60 (148.5 MHz), which is inside that range.
mode = VideoMode(1920, 1080, 24, 60)
v.hdmi_out.configure(mode)
v.hdmi_out.start()
print("hdmi_out configured:", v.hdmi_out.mode)

# HPD must be asserted or the receiving side never even reads the EDID.
hpd = None
for name in ol.ip_dict:
    if "gpio_hdmiin" in name.lower():
        hpd = getattr(ol, name)
        hpd.mmio.write(0x00, 0x1)
        print("HPD asserted on", name)

# A solid green frame, so if a capture card is ever put on this loop the
# picture is unmistakable.
frame = v.hdmi_out.newframe()
frame[:] = (0, 255, 0)
v.hdmi_out.writeframe(frame)
print("green frame written to the output")


## 监视输入是否锁定（20 秒）

In [ ]:
locked = None
for i in range(20):
    m = v.hdmi_in.mode
    if m is not None:
        locked = m
        print(f"[{i:2d}] LOCKED {m.width}x{m.height} @{m.frame_rate} Hz")
        break
    if i % 4 == 0:
        print(f"[{i:2d}] mode: None")
    time.sleep(1.0)

print()
if locked is None:
    print("RESULT: 自环也没锁上 -> 板子的 HDMI 输入通道有问题")
    print("  下一步：确认线确实接在 OUT 和 IN 之间；换一根线再试；")
    print("  若仍不行，这属于硬件问题，与算法和位流无关")
else:
    print(f"RESULT: 自环锁定成功 {locked.width}x{locked.height}@{locked.frame_rate}")
    print("  -> 板子的 HDMI 输入硬件是好的，问题在笔记本送出的信号")
    print("  -> 把笔记本接到 IN，分辨率设 1920x1080（或 1280x720）再试")


## 测完之后

把线恢复成原来的接法（笔记本→IN，OUT→采集卡），然后重新加载我们的位流：

```python
from pynq import Overlay
ol = Overlay("enhance_hdmi.bit")
```

或者直接重跑 `hdmi_after_fix.py`。
